Preparation file - cleaning, merging, dropping, whatever is needed before the datasets meet models

Run on python 3.13.13 kernel, on Visual studios code, using juypter notebook.
Uses the 1_Enron_ml_ready.csv and the 2_Naz_ml_ready.csv Designed to combine 1 of the two datasets with the 3_AI_phish_preprocess.csv
creates 2 stop gap datasets; first being 3_AI_phish_preprocess_V2.csv this is so that we can combine the altered AI phishing dataset with one of the other two, without saving over the original preprocessed file.
From here the combination 3_AI_phish_mix.csv is then processed and the finalised dataset 4_FINAL_preppared_dataset_(E or N for what dataset it contains).csv is then saved to be used in the ML models file.

Loading the dataset - AI, benign, phish or mix

In [13]:
import pandas as pd
# Load the dataset and display first rows
df = pd.read_csv("3_AI_phish_preprocess.csv", encoding="latin1")
print(df.head())

                                          ï»¿subject  \
0   A quick opportunity that deserves your attention   
1  Review recommended: a decision with financial ...   
2                I didnât want this to pass you by   
3            Your confirmation will keep this moving   
4  A quick booking confirmation while slots remai...   

                                          email_body     label  
0  Dear Pall,\n\nThis note is to ensure you have ...  AI_phish  
1  Hi Wally,\n\nIâm reaching out because there ...  AI_phish  
2  Hi Chryste,\n\nIâm reaching out because ther...  AI_phish  
3  Hi Josh,\n\nWe are contacting you regarding a ...  AI_phish  
4  Hi Joni,\n\nThis note is to ensure you have se...  AI_phish  


Dropping unnecessary columns and merging the subject and body

In [14]:
#encoding issues identified in the AI-generated phishing dataset
#checks data is string then resolves latin1 and cp1252
#differences in encoding
def fix_mojibake(text):
    if isinstance(text, str):
        try:
            return text.encode("latin1").decode("cp1252")
        except (UnicodeEncodeError, UnicodeDecodeError):
            return text
    return text

#applies the fix to dataframe
df.columns = df.columns.str.strip()
df.columns = df.columns.str.replace("ï»¿", "", regex=False)
print(df.columns.tolist())
df["subject"] = df["subject"].apply(fix_mojibake)
df["email_body"] = df["email_body"].apply(fix_mojibake)

#formatting the AI-generated emails in the same way as previous 2
df = df[["label", "subject", "email_body"]]
df["text"] = df["subject"] + " " + df["email_body"]
df = df[["label", "text"]]

df.to_csv("3_AI_phish_preprocess_V2.csv", index=False, encoding="utf-8-sig")

['subject', 'email_body', 'label']


merging the dataset

In [15]:
#Change based on whether using Enron or Naz
df1 = pd.read_csv("2_Naz_ml_ready.csv")

df2 = pd.read_csv("3_AI_phish_preprocess_V2.csv")

# Combine rows
combined_df = pd.concat([df1, df2], ignore_index=True)

print(combined_df.head())
combined_df.to_csv("3_AI_phish_mix.csv", index=False)

                                                text     label
0  Verify Your Account Business with cPanel & WHM...  phishing
1  Helpdesk Mailbox Alert!!! Your two incoming ma...  phishing
2  IT-Service Help Desk Password will expire in 3...  phishing
3  Final USAA Reminder - Update Your Account Now ...  phishing
4  PayPal Secure Dear Client, We have noticed tha...  phishing


checking the results of the merge

In [16]:
df = pd.read_csv("3_AI_phish_mix.csv")
print(df.head())

                                                text     label
0  Verify Your Account Business with cPanel & WHM...  phishing
1  Helpdesk Mailbox Alert!!! Your two incoming ma...  phishing
2  IT-Service Help Desk Password will expire in 3...  phishing
3  Final USAA Reminder - Update Your Account Now ...  phishing
4  PayPal Secure Dear Client, We have noticed tha...  phishing


Getting the word count average between classes

In [17]:
# Define columns
text_col = "text"
label_col = "label"

# Compute word count per row
df["word_count"] = df[text_col].fillna("").apply(lambda x: len(str(x).split()))

# Group by label and calculate average word count
avg_word_count = df.groupby(label_col)["word_count"].mean()

# Display result
print(avg_word_count)

label
AI_phish     99.713000
phishing    188.414322
Name: word_count, dtype: float64


installing model explainer

In [18]:
# commented out to prevent being installed on multiple runs

#!pip install shap
#!pip install spacy scikit-learn pandas
#!python -m spacy download en_core_web_sm

Explainer model running on Dataset

Imports and dataset loading

In [19]:
import pandas as pd
import numpy as np
import re
import unicodedata
import spacy

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, accuracy_score, confusion_matrix

# load spacy
nlp = spacy.load("en_core_web_sm", disable=["parser"])

# checking dataset size
print("Original dataset shape:", df.shape)

Original dataset shape: (2564, 3)


Basic cleaning

In [20]:
# basic row cleanup
# Remove nulls
df = df.dropna(subset=["text", "label"])

# Strip whitespace
df["text"] = df["text"].astype(str).str.strip()
df["label"] = df["label"].astype(str).str.strip()

# Remove blank rows
df = df[(df["text"] != "") & (df["label"] != "")]

# Remove duplicates
df = df.drop_duplicates()

print("After null/blank/duplicate removal:", df.shape)

# checking Label distribution
label_counts = df["label"].value_counts()

print("\nLabel distribution:")
print(label_counts)

# percentage breakdown
label_percent = df["label"].value_counts(normalize=True) * 100

print("\nLabel distribution (%):")
print(label_percent.round(2))

After null/blank/duplicate removal: (2552, 3)

Label distribution:
label
phishing    1552
AI_phish    1000
Name: count, dtype: int64

Label distribution (%):
label
phishing    60.82
AI_phish    39.18
Name: proportion, dtype: float64


Preprocessing

In [21]:
nlp = spacy.load("en_core_web_sm")
#words that should not be removed by spacy
#ensuring they dont get included as stop words
IMPORTANT_WORDS_TO_KEEP = {
    "account", "bank", "click", "confirm", "credential", "credentials",
    "email", "invoice", "link", "login", "password", "payment", "review",
    "security", "signin", "urgent", "verify", "warning", "wire", "update",
    "today", "now", "immediately", "message", "help"
}
#same but for organisations
IMPORTANT_ORGS = {
    "paypal", "amazon", "microsoft", "apple", "google", "bank",
    "chase", "wells", "fargo", "outlook", "office", "365"
}
#same but for headers
HEADER_NOISE = {
    "cc", "bcc", "subject", "forwarded", "from", "to", "date", "sent",
    "re", "fw", "fwd"
}
#identifying greeting/signoff words
GREETING_WORDS = {"hi", "hello", "dear", "hey"}
SIGNOFF_WORDS = {"regards", "best", "thanks", "sincerely", "cheers", "luck"}

#standardising the text, allowing for consistency
def normalize_unicode(text):
    text = unicodedata.normalize("NFKD", str(text))
    return text.encode("ascii", "ignore").decode("ascii")

#replacing names with PERSON to tackle enron dataset emails threads
def remove_person_spans(text):
    doc = nlp(text)
    spans = [(ent.start_char, ent.end_char) for ent in doc.ents if ent.label_ == "PERSON"]

    if not spans:
        return text

    cleaned = []
    last = 0
    #going through and swapping the names for stand in
    for start, end in sorted(spans):
        cleaned.append(text[last:start])
        cleaned.append(" PERSON ")
        last = end
    cleaned.append(text[last:])
    return "".join(cleaned)

#detecting names used in patterns and removes them
#Enron email thread problems
def remove_greeting_and_signoff_names(text):
    patterns = [
        r"\b(?:hi|hello|dear|hey)\s+[A-Z][a-z]+\b",
        r"\b(?:regards|best|thanks|sincerely|cheers)\s*,?\s+[A-Z][a-z]+\b",
        r"\bgood luck\s+[A-Z][a-z]+\b",
    ]
    for pat in patterns:
        text = re.sub(pat, " PERSON ", text)
    return text

#removing any other name like text
def remove_remaining_name_like_tokens(text):
    #run spaCy with text to get tokens, POS tags and names entities
    doc = nlp(text)
    kept = []

    #examine each token in sequence
    for i, token in enumerate(doc):
        tok = token.text
        low = tok.lower()

        #skip empty or whitespace tokens
        if not tok.strip():
            continue

        # Remove explicit person entities
        if token.ent_type_ == "PERSON":
            continue

        # Remove proper nouns unless told not to before hand
        if (
            token.pos_ == "PROPN"
            and re.fullmatch(r"[A-Z][a-z]+", tok)
            and low not in IMPORTANT_ORGS
        ):
            continue

        # Remove names after greetings/signoffs
        if i > 0:
            prev = doc[i - 1].text.lower().strip(",:")
            if prev in GREETING_WORDS or prev in SIGNOFF_WORDS:
                if re.fullmatch(r"[A-Z][a-z]+", tok):
                    continue
        #keep other tokens
        kept.append(tok)
    #rebuild the text
    return " ".join(kept)

#text normalisation before regex
def pre_regex_cleanup(text):
    text = normalize_unicode(text)

    #removing names multiple layers
    text = remove_greeting_and_signoff_names(text)
    text = remove_person_spans(text)
    text = remove_remaining_name_like_tokens(text)

    #to lowercase
    text = text.lower()

    # Remove leftover person placeholder
    text = re.sub(r"\bperson\b", " ", text)

    # ensuring no potential AI-generated obviously fake URLs remain by removing all
    # Replace URLs and emails
    text = re.sub(r"(https?://\S+|www\.\S+)", " URL ", text)
    text = re.sub(r"\b[\w\.-]+@[\w\.-]+\.\w+\b", " EMAIL ", text)

    # Normalize headers
    text = re.sub(r"\b(from|to|cc|bcc|subject|date|sent):\s*", " ", text)

    # Normalize numbers
    text = re.sub(r"\b\d+\b", " NUM ", text)

    # Keep useful chars
    text = re.sub(r"[^a-z0-9\s!\?\%\$\.\:/@]", " ", text)

    # Normalize punctuation
    text = re.sub(r"!{2,}", " !! ", text)
    text = re.sub(r"\?{2,}", " ?? ", text)

    # Collapse spaces
    text = re.sub(r"\s+", " ", text).strip()
    return text

#cleaning using spaCy
def spacy_clean_text(text, max_chars=200000):
    text = str(text)

    if len(text) > max_chars:
        text = text[:max_chars]

    text = pre_regex_cleanup(text)
    #tokenizes the text
    doc = nlp(text)
    cleaned_tokens = []
    #hard coded names that frequently appeared in Enron dataset or are common names
    COMMON_FIRST_NAMES = {
        "phillip", "philip", "mark", "john", "mary", "james", "robert",
        "michael", "david", "sarah", "jennifer", "linda", "william"
    }
    #filter tokens step by step
    for token in doc:
        tok = token.text.strip()
        lemma = token.lemma_.strip().lower()

        if not tok:
            continue
        #preserve special tokens
        if tok in {"URL", "EMAIL", "NUM", "!!", "??"}:
            cleaned_tokens.append(tok.lower())
            continue

        #remove whitespace and punctuation
        if token.is_space or token.is_punct:
            continue
        # remove header noise
        if tok.lower() in HEADER_NOISE:
            continue

        # remove any remaining names
        if token.ent_type_ == "PERSON":
            continue

        if token.pos_ == "PROPN" and tok.lower() not in IMPORTANT_ORGS:
            continue

        #remove stop words
        if token.is_stop and lemma not in IMPORTANT_WORDS_TO_KEEP:
            continue

        #lemmatization
        final_tok = lemma if lemma not in {"-pron-"} else tok.lower()

        # Remove common first names
        if final_tok in COMMON_FIRST_NAMES:
            continue
        #regex filter
        if re.fullmatch(r"[a-z0-9!?\%\$@:/\.]+", final_tok):
            cleaned_tokens.append(final_tok)

    return " ".join(cleaned_tokens)


#Apply cleaning to text
df["clean_text"] = df["text"].apply(spacy_clean_text)

# Drop empty rows
df = df[df["clean_text"].str.strip() != ""].copy()
df.reset_index(drop=True, inplace=True)

print("After preprocessing:", df.shape)
print(df[["text", "clean_text"]].head(5).to_string(index=False))

#save the prepared data frame
#would recommend renaming if running the other dataset
#with the AI-phish, to prevent overwriting
df.to_csv("4_FINAL_prepared_dataset_N.csv", index=False, encoding="utf-8-sig")

After preprocessing: (2547, 4)
                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                         

Now that the text has been cleaned, it can be run through the tester model to see if there is other key words that sway predictions

Train/Test split

In [22]:
# train and test split
X = df["clean_text"]
y = df["label"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

Explainer LR model

In [23]:
#Building the model
model = Pipeline([
    ("tfidf", TfidfVectorizer(
        max_features=5000, # top 5k features
        ngram_range=(1, 2), # use single words and 2-word phrases
        min_df=2, # ignore words that appear only once
        sublinear_tf=True # reduce words that appear once
    )),
    ("clf", LogisticRegression(
        max_iter=2000,
        random_state=42,
        class_weight="balanced"
    ))
])

# training
model.fit(X_train, y_train)

# evaluating
y_pred = model.predict(X_test)

print("\nAccuracy:")
print(round(accuracy_score(y_test, y_pred), 4))

print("\nClassification report:")
print(classification_report(y_test, y_pred))

print("\nConfusion matrix:")
print(confusion_matrix(y_test, y_pred))

# model components
vectorizer = model.named_steps["tfidf"]
clf = model.named_steps["clf"]
feature_names = vectorizer.get_feature_names_out()
classes = clf.classes_


Accuracy:
1.0

Classification report:
              precision    recall  f1-score   support

    AI_phish       1.00      1.00      1.00       200
    phishing       1.00      1.00      1.00       310

    accuracy                           1.00       510
   macro avg       1.00      1.00      1.00       510
weighted avg       1.00      1.00      1.00       510


Confusion matrix:
[[200   0]
 [  0 310]]


model explanation of decisions

In [ ]:
# global explanation of the text - what pushed a decision across
# the entire group

#title of what explainer this is
def show_top_features_per_class(top_n=20):
    print("GLOBAL MODEL EXPLANATION")

    #binary classification case (since using LR)
    if len(classes) == 2 and clf.coef_.shape[0] == 1:
        #get coefficients
        coefs = clf.coef_[0]

        #top positive words
        top_positive_idx = np.argsort(coefs)[-top_n:][::-1]
        #top negative words 
        top_negative_idx = np.argsort(coefs)[:top_n]

        #printing out the top words that pushed towards and
        #away from a prediction
        print(f"\nWords pushing toward class: {classes[1]}")
        for idx in top_positive_idx:
            print(f"  {feature_names[idx]:<30} {coefs[idx]:.4f}")

        print(f"\nWords pushing toward class: {classes[0]}")
        for idx in top_negative_idx:
            print(f"  {feature_names[idx]:<30} {coefs[idx]:.4f}")

    else:
        #loop through each class and get class specific coefficients
        for class_idx, class_name in enumerate(classes):
            coefs = clf.coef_[class_idx]
            #top words per class
            top_idx = np.argsort(coefs)[-top_n:][::-1]

            print(f"\nTop words for class: {class_name}")
            for idx in top_idx:
                print(f"  {feature_names[idx]:<30} {coefs[idx]:.4f}")
#show the top 20 words
show_top_features_per_class(top_n=20)

#local explainer - what pushed decision in one example
def explain_prediction(raw_text, top_n=10):
    #ensuring the prediction is run on same preprocessing
    cleaned_text = spacy_clean_text(raw_text)

    #predicting the class
    pred = model.predict([cleaned_text])[0]
    #probabilities for each class
    probs = model.predict_proba([cleaned_text])[0]

    #printing the title
    print("LOCAL PREDICTION EXPLANATION")

    #showing the original and cleaned text
    print("\nOriginal input text:")
    print(str(raw_text)[:1000])

    #show the predicted class
    print("\nCleaned input text:")
    print(cleaned_text[:1000])

    #show the probabilities
    print("\nPredicted label:", pred)

    print("\nClass probabilities:")
    for class_name, prob in zip(classes, probs):
        print(f"  {class_name:<20} {prob:.4f}")

    #vectorize clean text -  turn to numerical feature vector
    x_vec = vectorizer.transform([cleaned_text]).toarray()[0]
    #getting the indices of features that are in the actual text
    nonzero_idx = np.where(x_vec > 0)[0]

    #check the model is binary
    if len(classes) == 2 and clf.coef_.shape[0] == 1:
        #compute the contributions check how much a feature appears
        #and how strongly that feature affects the prediction
        contributions = x_vec * clf.coef_[0]

        #seeing if the predicted class is the positive class
        if pred == classes[1]:
            ranked_idx = nonzero_idx[np.argsort(contributions[nonzero_idx])[-top_n:]][::-1]
        else:
            #seeing if the predicted class is the negative class
            ranked_idx = nonzero_idx[np.argsort(contributions[nonzero_idx])[:top_n]]
    else:
        #gets the position of the predicted class in class list
        pred_idx = list(classes).index(pred)
        #compute the class specific contributions
        contributions = x_vec * clf.coef_[pred_idx]
        #rank the top contributors
        ranked_idx = nonzero_idx[np.argsort(contributions[nonzero_idx])[-top_n:]][::-1]

    #print the top contributing words and their values
    print(f"\nTop words pushing toward predicted class: {pred}")
    for idx in ranked_idx:
        print(f"  {feature_names[idx]:<30} {contributions[idx]:.4f}")
        
#grabs a test example and explains the prediction
def explain_test_example(example_index=0, top_n=10):
    #get raw text
    raw_text = df.loc[X_test.index[example_index], "text"]
    #actual label for the example
    true_label = y_test.iloc[example_index]

    #print the real label to compare to predicted
    print("\nTrue label:", true_label)
    explain_prediction(raw_text, top_n=top_n)
#show the first test example and top 10 most influential words
explain_test_example(example_index=0, top_n=10)

# simplifing the explanation
def plain_english_explanation(raw_text, top_n=8):
    #clean text
    cleaned_text = spacy_clean_text(raw_text)
    #predict labels and probabilities
    pred = model.predict([cleaned_text])[0]
    probs = model.predict_proba([cleaned_text])[0]

    #vectorization
    x_vec = vectorizer.transform([cleaned_text]).toarray()[0]
    nonzero_idx = np.where(x_vec > 0)[0]

    #computing the contributions
    if len(classes) == 2 and clf.coef_.shape[0] == 1:
        contributions = x_vec * clf.coef_[0]
        if pred == classes[1]:
            ranked_idx = nonzero_idx[np.argsort(contributions[nonzero_idx])[-top_n:]][::-1]
        else:
            ranked_idx = nonzero_idx[np.argsort(contributions[nonzero_idx])[:top_n]]
    else:
        pred_idx = list(classes).index(pred)
        contributions = x_vec * clf.coef_[pred_idx]
        ranked_idx = nonzero_idx[np.argsort(contributions[nonzero_idx])[-top_n:]][::-1]
    #get the top words
    top_words = [feature_names[idx] for idx in ranked_idx]

    #printing the explanation
    print("PLAIN-ENGLISH SUMMARY")
    #the explanation
    print(
        f"The model predicted '{pred}' because the cleaned text contains signals "
        f"it has learned to associate with that class."
    )
    #key signals or the most influential words
    print(f"Strongest signals: {', '.join(top_words)}")
    print("\nConfidence by class:")
    for class_name, prob in zip(classes, probs):
        print(f"  {class_name}: {prob:.4f}")


GLOBAL MODEL EXPLANATION

Words pushing toward class: phishing
  email                          1.5695
  account                        1.3776
  click                          1.1792
  mail                           1.1538
  information                    1.0053
  message                        0.8792
  receive                        0.8471
  update                         0.7080
  new                            0.6917
  utf                            0.6886
  service                        0.6884
  upgrade                        0.6858
  link                           0.6795
  payment                        0.6444
  expire                         0.6359
  view                           0.6257
  monkey                         0.6102
  security                       0.6079
  mailbox                        0.6077
  password                       0.6053

Words pushing toward class: AI_phish
  now                            -2.2213
  quick                          -2.0714
  review         